# 02.1 — Loading and chunking

The first step of any RAG pipeline is turning documents into pieces small
enough to retrieve. This notebook does that in the most obvious way possible,
then looks closely at the result.

The point is not to do it well. It's to see exactly how the obvious approach
fails, so that the fixes in later modules have something to fix.

In [1]:
!pip install -q pymupdf4llm==1.28.2

In [2]:
from pathlib import Path
import pymupdf4llm

CORPUS = Path('../../corpus/docs')
assert CORPUS.exists(), f'corpus not found at {CORPUS.resolve()}'
print(CORPUS.resolve())

/home/jovyan/work/corpus/docs


## What's in the corpus

Fifteen documents from three fictional organisations. They're
synthetic, but they behave like real business documents — which means most of
them are awkward in some specific way.

`corpus/README.md` explains what's wrong with each one. Worth reading, but not
yet — it's more useful after you've hit the problems yourself.

In [3]:
for p in sorted(CORPUS.iterdir()):
    print(f'{p.suffix:<7} {p.name}')

.pdf    kaduna-agro-annual-report-2024.pdf
.pptx   kaduna-agro-board-deck-2025-01.pptx
.pdf    kaduna-agro-board-minutes-2024-10-17.pdf
.xlsx   kaduna-agro-distribution-2024.xlsx
.pdf    kdirs-guidance-note-4-2024-SCANNED.pdf
.pdf    nfsc-circular-2024-07-cybersecurity.pdf
.pdf    nfsc-circular-2025-02-amendment.pdf
.html   nfsc-circular-2025-02.html
.csv    sahel-approved-vendors.csv
.md     sahel-branch-runbook.md
.pdf    sahel-employee-handbook-2023.pdf
.pdf    sahel-employee-handbook-2025.pdf
.docx   sahel-hr-memo-2024-41-TRACKED.docx
.eml    sahel-per-diem-thread.eml
.pdf    sahel-procurement-policy-v3.pdf


## Today we read seven of them

Only the PDFs with an extractable text layer. We're skipping:

- the `.docx`, `.xlsx`, `.pptx`, `.html`, `.eml` and `.csv` files — each needs a
  different library
- `kdirs-guidance-note-4-2024-SCANNED.pdf` — a photocopy with no text in it at
  all, only pixels

That's not laziness. Roughly a third of the test questions in
`corpus/golden_questions.csv` point at documents this notebook cannot read, so
the score we produce in notebook 5 will be capped well below what's possible.

Module 03 handles the other formats, and you'll watch that number jump. If we
solved ingestion now, module 03 would have nothing to prove.

In [4]:
NATIVE_PDFS = [
    'sahel-employee-handbook-2023.pdf',
    'sahel-employee-handbook-2025.pdf',
    'sahel-procurement-policy-v3.pdf',
    'nfsc-circular-2024-07-cybersecurity.pdf',
    'nfsc-circular-2025-02-amendment.pdf',
    'kaduna-agro-annual-report-2024.pdf',
    'kaduna-agro-board-minutes-2024-10-17.pdf',
]

docs = {}
for name in NATIVE_PDFS:
    docs[name] = pymupdf4llm.to_markdown(str(CORPUS / name))

for name, text in docs.items():
    print(f'{len(text):>7,} chars   {name}')

print(f'\n{sum(len(t) for t in docs.values()):,} characters total')

  6,472 chars   sahel-employee-handbook-2023.pdf
  6,559 chars   sahel-employee-handbook-2025.pdf
  3,595 chars   sahel-procurement-policy-v3.pdf
  4,286 chars   nfsc-circular-2024-07-cybersecurity.pdf
  2,300 chars   nfsc-circular-2025-02-amendment.pdf
  6,120 chars   kaduna-agro-annual-report-2024.pdf
  4,057 chars   kaduna-agro-board-minutes-2024-10-17.pdf

33,389 characters total


`to_markdown()` gives us markdown rather than a wall of plain text, so headings
and tables survive as structure. Have a look at what came out.

`pymupdf4llm` prints its own progress messages, including one about Tesseract
and OCR. That's it deciding, on its own, to run optical character recognition on
part of a PDF that already has perfectly good text in it — the chart image in the
annual report.

OCR is roughly a thousand times slower than reading the text layer, and nothing
in the output tells you which text came from where. Ignore it for now. Module 03
is where that matters.

In [5]:
print(docs['sahel-employee-handbook-2025.pdf'][:700])

Employee Handbook — 2025 Edition 

# **Sahel Microfinance Bank Plc** 

Document reference: SMB/HR/HANDBOOK/2025  |  Effective 1 January 2025  |  Owner: Head, Human Capital  |  Classification: Internal 

## **1. Introduction** 

Sahel Microfinance Bank Plc ("the Bank") operates twenty-two branches across Lagos, Ogun, Kaduna, Kano and the Federal Capital Territory. This handbook sets out the terms of employment, the conduct expected of staff, and the benefits available to confirmed employees. It supersedes all previous editions and all informal practices at branch level. 

Where this handbook conflicts with an individual employment contract, the contract prevails. Where it conflicts with Niger


## Chunking, the obvious way

An embedding model can only handle a limited amount of text at once, and
retrieving a whole 6,000-character document to answer one question wastes most
of the context you send to the LLM. So documents get split.

The simplest possible split: every 500 characters, regardless of what's there.

500 is arbitrary. Module 04 is about choosing it properly.

In [6]:
def chunk(text, size=500):
    return [text[i:i + size] for i in range(0, len(text), size)]


chunks = []
for name, text in docs.items():
    for i, body in enumerate(chunk(text)):
        chunks.append({'doc': name, 'n': i, 'text': body})

print(f'{len(docs)} documents -> {len(chunks)} chunks')

7 documents -> 71 chunks


## Now look at what that did

Seventy-one chunks, produced in three lines. It ran without error, which is the
problem — nothing warns you when a split lands somewhere stupid.

### Failure one: it cuts through sentences

One of the test questions asks how much annual leave a confirmed staff member
gets. Here's where the answer ended up.

In [7]:
def show(doc, n):
    c = next(c for c in chunks if c['doc'] == doc and c['n'] == n)
    print(f'--- chunk {n} ---')
    print(c['text'])


show('sahel-employee-handbook-2025.pdf', 3)
show('sahel-employee-handbook-2025.pdf', 4)

--- chunk 3 ---
 in locations with Saturday banking operate a rota under which no member of staff works more than two Saturdays in any calendar month. Time off in lieu is granted at the branch manager's discretion and must be taken within the same quarter. 

Persistent lateness, defined as arrival after 08:15 on more than four occasions in a calendar month, is treated as a conduct matter under section 8. 

## **4. Annual leave** 

Confirmed staff are entitled to **25 working days** of paid annual leave each cal
--- chunk 4 ---
endar year, exclusive of public holidays. Leave accrues monthly and may be taken from the seventh month of service. A maximum of 10 working days may be carried into the following year and must be exhausted by 31 March, after which the balance lapses without payment in lieu. 

Leave applications require the approval of the line manager and, for absences exceeding ten consecutive working days, the approval of the divisional head. No more than two members of any bra

The sentence runs off the end of chunk 3 and continues in chunk 4. The word
`calendar` is split in half — `cal` at the end of one, `endar` at the start of
the next.

Retrieve chunk 4 and you get a rule with no subject. Retrieve chunk 3 and you
get an entitlement with no qualification. Neither is wrong exactly, and neither
is enough.

### Failure two: tables lose their headers

The annual report has a table of distribution points — state, local government
area, volume, revenue, utilisation. Watch what a fixed-size split does to it.

In [8]:
show('kaduna-agro-annual-report-2024.pdf', 7)

--- chunk 7 ---
---|
|Oyo|Ibadan North|31,487|17,136.7|80%|
|Oyo|Ogbomoso|44,403|19,978.0|41%|
|Oyo|Oyo Town|36,427|15,736.5|43%|
|Kaduna|Kaduna South|14,265|8,375.2|79%|
|Kaduna|Zaria|3,771|2,141.3|61%|
|Kaduna|Kafanchan|30,670|16,325.1|53%|
|Kano|Kano Municipal|35,821|16,635.0|59%|
|Kano|Dala|34,553|14,542.4|46%|
|Kano|Wudil|31,771|17,296.5|67%|
|Rivers|Port Harcourt|37,927|23,104.4|94%|
|Rivers|Obio-Akpor|7,252|4,021.1|61%|
|Rivers|Bonny|16,851|8,719.5|42%|
|Delta|Warri|6,402|3,373.8|47%|
|Delta|Asaba|28,040


Bare rows. Nothing in this chunk says that `32,316` is a volume in tonnes and
`95%` is capacity utilisation. An LLM handed this has no way to answer a
question about it, and worse, it might guess.

There's a second problem visible here, and it isn't ours. Look for the
`|---|---|---|---|---|` line partway down. That's a markdown header separator,
and the row above it — `|Ogun|Ijebu-Ode|43,379|...|` — has been turned into a
table header.

The table breaks across a page in the original PDF, and `pymupdf4llm` emitted
the second half as a brand new table, promoting the first data row to be its
header. Every row after that point has lost its real column names, and the
output looks perfectly well-formed.

Nothing errored. This is the shape of most RAG bugs: not a crash, just quietly
wrong data that flows all the way through to a confident answer.

## What's next

You now have 71 chunks with at least two known defects. Notebook 2 turns them
into vectors and searches them.

Don't fix anything yet. Both failures above are the subject of module 04, and
the missing documents are module 03. First we need a way to measure how much
they actually cost us — that's notebook 5.